In [ ]:
cd ..

In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from src.saferesponse_engine import logger
from src.saferesponse_engine.config.configuration import ConfigurationManager
from src.saferesponse_engine.components.fusion_decision_router import FusionDecisionRouter

In [ ]:
@dataclass(frozen=True)
class FusionRouterResearchConfig:
    root_dir: Path
    verification_artifact_path: Path
    traces_artifact_path: Path
    fusion_output_path: Path
    weight_halluguard: float
    weight_grounding: float
    weight_consistency: float
    weight_judge: float
    accept_threshold: float
    rewrite_threshold: float
    reject_threshold: float
    max_rewrite_attempts: int

In [ ]:
base_config = ConfigurationManager().get_fusion_router_config()

fusion_config = FusionRouterResearchConfig(
    root_dir=base_config.root_dir,
    verification_artifact_path=base_config.verification_artifact_path,
    traces_artifact_path=base_config.traces_artifact_path,
    fusion_output_path=base_config.fusion_output_path,
    weight_halluguard=base_config.weight_halluguard,
    weight_grounding=base_config.weight_grounding,
    weight_consistency=base_config.weight_consistency,
    weight_judge=base_config.weight_judge,
    accept_threshold=base_config.accept_threshold,
    rewrite_threshold=base_config.rewrite_threshold,
    reject_threshold=base_config.reject_threshold,
    max_rewrite_attempts=base_config.max_rewrite_attempts,
)

fusion_config

In [ ]:
def load_plain_json(path: Path) -> dict[str, Any]:
    with Path(path).open("r", encoding="utf-8") as file:
        return json.load(file)


verification_data = load_plain_json(fusion_config.verification_artifact_path)
trace_data = load_plain_json(fusion_config.traces_artifact_path)

query = verification_data["query"]
candidates = verification_data.get("candidates", [])
traces = {
    trace["response_id"]: trace
    for trace in trace_data.get("traces", [])
}

query, len(candidates), sorted(traces)

In [ ]:
def clip(value: float, low: float = 0.0, high: float = 1.0) -> float:
    return max(low, min(high, value))


def as_float(value: Any, default: float) -> float:
    if value is None:
        return default
    try:
        return float(value)
    except (TypeError, ValueError):
        return default


def fuse_scores(candidate: dict[str, Any]) -> dict[str, Any]:
    halluguard_score = clip(as_float(candidate.get("halluguard_score"), 1.0))
    grounding_score = clip(as_float(candidate.get("grounding_score"), 0.0))
    consistency_score = clip(as_float(candidate.get("consistency_score"), 0.0))
    judge_score = candidate.get("judge_score")

    risk_terms = {
        "halluguard_risk": halluguard_score,
        "grounding_risk": 1.0 - grounding_score,
        "consistency_risk": 1.0 - consistency_score,
    }
    weighted_terms = [
        ("halluguard", risk_terms["halluguard_risk"], fusion_config.weight_halluguard),
        ("grounding", risk_terms["grounding_risk"], fusion_config.weight_grounding),
        ("consistency", risk_terms["consistency_risk"], fusion_config.weight_consistency),
    ]

    if judge_score is not None:
        risk_terms["judge_risk"] = clip(as_float(judge_score, 1.0))
        weighted_terms.append(("judge", risk_terms["judge_risk"], fusion_config.weight_judge))
    else:
        risk_terms["judge_risk"] = None

    weight_sum = sum(weight for _, _, weight in weighted_terms if weight > 0)
    combined_risk = sum(
        (weight / weight_sum) * risk
        for _, risk, weight in weighted_terms
        if weight > 0
    ) if weight_sum > 0 else 1.0

    effective_weights = {
        name: round(weight / weight_sum, 6)
        for name, _, weight in weighted_terms
        if weight > 0 and weight_sum > 0
    }

    return {
        "combined_risk": round(clip(combined_risk), 6),
        "risk_terms": {
            key: round(value, 6) if isinstance(value, float) else value
            for key, value in risk_terms.items()
        },
        "effective_weights": effective_weights,
    }

In [ ]:
def build_fusion_scores(
    candidates: list[dict[str, Any]],
    traces: dict[int, dict[str, Any]],
) -> list[dict[str, Any]]:
    fusion_scores = []
    for candidate in candidates:
        response_id = candidate["response_id"]
        trace = traces.get(response_id, {})
        fused = fuse_scores(candidate)
        fusion_scores.append({
            "response_id": response_id,
            "text": candidate.get("text", ""),
            "is_primary": bool(candidate.get("is_primary", response_id == 0)),
            "combined_risk": fused["combined_risk"],
            "risk_terms": fused["risk_terms"],
            "effective_weights": fused["effective_weights"],
            "sequence_score": trace.get("sequence_score"),
            "halluguard_score": candidate.get("halluguard_score"),
            "grounding_score": candidate.get("grounding_score"),
            "consistency_score": candidate.get("consistency_score"),
            "judge_score": candidate.get("judge_score"),
            "risk_signals": candidate.get("risk_signals", {}),
            "supporting_source": candidate.get("supporting_source", {}),
        })

    ranked = sorted(
        fusion_scores,
        key=lambda item: (
            item["combined_risk"],
            -as_float(item.get("sequence_score"), float("-inf")),
        ),
    )
    for rank, candidate in enumerate(ranked, start=1):
        candidate["rank"] = rank
    return ranked

In [ ]:
def route_candidate(
    best: dict[str, Any],
    rewrite_attempt: int = 0,
) -> tuple[str, str]:
    risk = best["combined_risk"]
    response_id = best["response_id"]
    is_primary = bool(best.get("is_primary", response_id == 0))

    if risk < fusion_config.accept_threshold:
        return (
            "ACCEPT",
            f"Candidate {response_id} combined_risk {risk:.3f} < accept_threshold {fusion_config.accept_threshold:.3f}.",
        )

    if risk < fusion_config.rewrite_threshold:
        if not is_primary:
            return (
                "RERANK",
                f"Candidate {response_id} is safer than the primary candidate and combined_risk {risk:.3f} < rewrite_threshold {fusion_config.rewrite_threshold:.3f}.",
            )
        return (
            "ACCEPT",
            f"Primary candidate is best available and combined_risk {risk:.3f} < rewrite_threshold {fusion_config.rewrite_threshold:.3f}.",
        )

    if risk < fusion_config.reject_threshold:
        if rewrite_attempt < fusion_config.max_rewrite_attempts:
            return (
                "REWRITE",
                f"Best candidate combined_risk {risk:.3f} requires rewrite attempt {rewrite_attempt + 1} of {fusion_config.max_rewrite_attempts}.",
            )
        return (
            "REJECT",
            f"Best candidate combined_risk {risk:.3f} is below reject threshold, but max rewrite attempts are exhausted.",
        )

    return (
        "REJECT",
        f"Best candidate combined_risk {risk:.3f} >= reject_threshold {fusion_config.reject_threshold:.3f}.",
    )


def build_rewrite_query(
    query: str,
    decision: str,
    selected_candidate: dict[str, Any],
) -> str | None:
    if decision != "REWRITE":
        return None
    source = selected_candidate.get("supporting_source", {}).get("source")
    if not source:
        return query
    return f"{query}\n\nFocus specifically on: {source}"

In [ ]:
def run_stage6_research(rewrite_attempt: int = 0, write_artifact: bool = True) -> dict[str, Any]:
    verification_data = load_plain_json(fusion_config.verification_artifact_path)
    trace_data = load_plain_json(fusion_config.traces_artifact_path)
    query = verification_data["query"]
    candidates = verification_data.get("candidates", [])
    traces = {
        trace["response_id"]: trace
        for trace in trace_data.get("traces", [])
    }
    if not candidates:
        raise ValueError("No verified candidates found for Stage 6 routing.")

    ranked = build_fusion_scores(candidates, traces)
    best = ranked[0]
    decision, decision_reason = route_candidate(best, rewrite_attempt=rewrite_attempt)
    rewrite_query = build_rewrite_query(query, decision, best)

    output = {
        "query": query,
        "decision": decision,
        "selected_response_id": best["response_id"],
        "rewrite_attempt": rewrite_attempt,
        "decision_reason": decision_reason,
        "thresholds": {
            "accept_threshold": fusion_config.accept_threshold,
            "rewrite_threshold": fusion_config.rewrite_threshold,
            "reject_threshold": fusion_config.reject_threshold,
            "max_rewrite_attempts": fusion_config.max_rewrite_attempts,
        },
        "configured_weights": {
            "halluguard": fusion_config.weight_halluguard,
            "grounding": fusion_config.weight_grounding,
            "consistency": fusion_config.weight_consistency,
            "judge": fusion_config.weight_judge,
        },
        "source_artifacts": {
            "verification": str(fusion_config.verification_artifact_path),
            "traces": str(fusion_config.traces_artifact_path),
        },
        "fusion_scores": ranked,
        "selected_candidate": {
            "response_id": best["response_id"],
            "text": best["text"],
            "is_primary": best["is_primary"],
            "combined_risk": best["combined_risk"],
            "rank": best["rank"],
            "supporting_source": best["supporting_source"],
            "risk_signals": best["risk_signals"],
        },
        "rewrite_query": rewrite_query,
    }

    if write_artifact:
        fusion_config.fusion_output_path.parent.mkdir(parents=True, exist_ok=True)
        fusion_config.fusion_output_path.write_text(
            json.dumps(output, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        logger.info("[Stage 6 Research] Fusion artifact saved: %s", fusion_config.fusion_output_path)

    return output


fusion_result = run_stage6_research()
fusion_result["decision"], fusion_result["selected_response_id"], fusion_result["selected_candidate"]["combined_risk"]

In [ ]:
[
    {
        "rank": candidate["rank"],
        "response_id": candidate["response_id"],
        "combined_risk": candidate["combined_risk"],
        "effective_weights": candidate["effective_weights"],
        "source": candidate.get("supporting_source", {}).get("source"),
    }
    for candidate in fusion_result["fusion_scores"]
]

In [ ]:
component_router = FusionDecisionRouter(config=base_config)
component_result = component_router.route()
component_result["decision"], component_result["selected_response_id"], component_result["selected_candidate"]["combined_risk"]